# Gold: Sales Global

**Objetivo:** consolidar o total de vendas globais (todos os países somados) por dia, apenas em USD — visão executiva de topo, para o CFO acompanhar a receita consolidada da operação LATAM como um todo, sem quebra por país.

**Fonte:** tabela Delta `silver.fato_vendas`.

**Destino:** tabela Delta `gold.sales_global`.

**Granularidade:** dia (`period`), consolidado (sem quebra por país).

**Observação:** como a soma é feita exclusivamente em USD (não faz sentido somar moedas diferentes sem conversão), dias sem cotação de câmbio disponível resultam em `total_usd` nulo.

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, sum as spark_sum, round as spark_round

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.gold")

print("Schema 'gold' verificado/criado com sucesso.")

In [0]:
# Agregação global por dia (somente USD)

df_fato_vendas = spark.table("poc_latam_food.silver.fato_vendas")

df_gold_global = (
    df_fato_vendas
    .groupBy(col("data_venda").alias("period"))
    .agg(
        spark_round(spark_sum("valor_total_usd"), 2).cast("decimal(15,2)").alias("total_usd")
    )
    .orderBy("period")
)

df_gold_global.display()

In [0]:
# Gravação - overwrite completo (recálculo)

(
    df_gold_global
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("poc_latam_food.gold.sales_global")
)

print("Gold sales_global concluído.")

In [0]:
# Validação visual - gold.sales_global

spark.table("poc_latam_food.gold.sales_global").display()